In [1]:
import os

from langchain_community.document_loaders import TextLoader
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from transformers import pipeline
from langchain_text_splitters import MarkdownHeaderTextSplitter

C:\Users\U S E R\AppData\Local\Temp\ipykernel_14920\1597996319.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
from pathlib import Path

project_root = Path.cwd().parent
file_path = project_root / "data" / "tennis_details.md"

print(file_path)

c:\Users\U S E R\github_projects\ai-engineering-lab\data\tennis_details.md


In [ ]:
#step 1 loading the data into TextLoader
loader = TextLoader(file_path)
text_doc = loader.load()
print(type(text_doc[0].metadata))
print(type(text_doc[0].page_content))
print(text_doc[0].metadata["source"])
#print(text_doc[0].page_content)
#print(text_doc[0]["source"])

<class 'dict'>
<class 'str'>
c:\Users\U S E R\github_projects\ai-engineering-lab\data\tennis_details.md


In [46]:
from langchain_core.documents import Document

In [53]:
#divide the data into chunks
split_condition = [("##", "title")]
splitter = MarkdownHeaderTextSplitter(split_condition)
doc_splits = splitter.split_text(text_doc[0].page_content)

final_documents = []
for split in doc_splits:
    if split.metadata:
        combined_text = f"{split.metadata['title']}\n\n{split.page_content}"

        doc = Document(
        page_content=combined_text,
        metadata={"title": split.metadata["title"]}
        )

        final_documents.append(doc)
    
    else:
        continue

## now we don't want title and metadata we just want page content 
text_chunks = [chunk.page_content for chunk in final_documents]
print(text_chunks) 
print(len(text_chunks))

["Introduction\n\nTennis is a popular sport played between two players (singles) or two teams of two players each (doubles). The game involves using a racket to hit a ball over a net into the opponent's court.", 'Basic Rules\n\n- A match can be played as best of three or five sets.\n- Each set consists of games, and each game consists of points.\n- Points are scored as **0 (Love), 15, 30, 40**, and then **game**.\n- A player must win a game by at least **two points**.\n- The ball must land within the designated court boundaries.', 'Scoring System\n\n```plaintext\n0 points  -> Love\n1 point   -> 15\n2 points  -> 30\n3 points  -> 40\n4 points  -> Game (if leading by 2)\nDeuce     -> 40-40 (must win two consecutive points to win the game)\nAdvantage -> If a player wins a point at deuce, they gain the advantage\n```', 'Famous Tournaments\n\n- **Grand Slam Events**:\n- Australian Open\n- French Open\n- Wimbledon\n- US Open', 'Equipment\n\n- **Racket**: Used to hit the ball.\n- **Tennis Ball

In [54]:
## step 3 now we need to use the embedding model 
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
def embed_chunks(chunk):
    return embedding_model.encode([chunk], normalize_embeddings=True).tolist()[0]

## let's pass some chunks manually 
sample_embedding = embed_chunks(text_chunks[1])
print(sample_embedding)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\U S E R\github_projects\ai-engineering-lab\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\U S E R\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[0.0366043858230114, 0.031236983835697174, -0.011291734874248505, -0.1105906069278717, -0.03547212481498718, 0.08182574808597565, -0.010864287614822388, 0.012413093820214272, 0.08294390887022018, 0.05625681206583977, -0.09208187460899353, -0.028846241533756256, 0.0486975759267807, 0.026132948696613312, 0.07727203518152237, 0.006983770988881588, 0.03473745286464691, -0.03163354471325874, -0.035051509737968445, 0.01874694973230362, 0.0776996836066246, -0.12796062231063843, -0.023384328931570053, -0.09138783812522888, -0.02452985569834709, 0.0035339791793376207, 0.022954890504479408, 0.032840192317962646, -0.06356234848499298, -0.016329756006598473, 0.028168581426143646, -0.053128525614738464, 0.06574724614620209, 0.0027148928493261337, -0.10749557614326477, -0.05431312322616577, -0.10580563545227051, -0.04896838963031769, -0.007547406945377588, 0.03872424364089966, 0.015896232798695564, -0.09384462982416153, 0.0432434119284153, 0.05760864168405533, 0.017019163817167282, 0.133166730403900

In [55]:
print(len(sample_embedding))

384


In [56]:
## store embedding in chroma db 
vector_db = Chroma.from_texts(text_chunks, HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2"), persist_directory="/tmp/chroma_db")
vector_db._collection.get(include=['embeddings','documents'])

C:\Users\U S E R\AppData\Local\Temp\ipykernel_14920\3012547257.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  vector_db = Chroma.from_texts(text_chunks, HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2"), persist_directory="/tmp/chroma_db")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

{'ids': ['fe25cf46-1989-4515-95e9-11f38a896858',
  '4006dbe5-63d2-41fc-b60c-f1ba255e0c8e',
  'ee3bfcf1-1e5f-4493-8241-767db8fdb08e',
  '0dcd6448-2c9a-46fd-8674-ce2a296be670',
  '622ec8d9-6ef6-4531-bf0d-f6b08e75109d',
  'cb0c8e08-5026-4afc-aacb-d4f73444d559'],
 'embeddings': array([[ 0.03047019,  0.01665993,  0.04339563, ...,  0.00187416,
          0.0035126 ,  0.04621071],
        [ 0.03660438,  0.03123698, -0.01129173, ...,  0.04196067,
         -0.01674688,  0.01525881],
        [-0.01810408,  0.0544811 , -0.0632717 , ...,  0.05674497,
         -0.01087213, -0.03204059],
        [ 0.04401479,  0.01293162,  0.01214257, ..., -0.07315321,
         -0.01768727,  0.00811605],
        [ 0.04375017,  0.05251511,  0.0314235 , ..., -0.05297537,
          0.04927486,  0.03786966],
        [ 0.04302077,  0.04445143,  0.07107276, ..., -0.00764226,
          0.00687301,  0.02607554]], shape=(6, 384)),
 'documents': ["Introduction\n\nTennis is a popular sport played between two players (singles) o

In [57]:
result = vector_db._collection.get(
    include=["embeddings", "documents"]
)
for i in range(len(result["documents"])):
    print("INDEX:", i)
    print("ID:", result["ids"][i])
    print("DOCUMENT:", result["documents"][i])
    print("EMBEDDING LENGTH:", len(result["embeddings"][i]))
    print("FIRST 10 VALUES:", result["embeddings"][i][:10])
    print("-" * 80)

INDEX: 0
ID: fe25cf46-1989-4515-95e9-11f38a896858
DOCUMENT: Introduction

Tennis is a popular sport played between two players (singles) or two teams of two players each (doubles). The game involves using a racket to hit a ball over a net into the opponent's court.
EMBEDDING LENGTH: 384
FIRST 10 VALUES: [ 0.03047019  0.01665993  0.04339563 -0.04162525 -0.11278321  0.03173719
  0.05185332  0.07799219  0.05089829  0.13183184]
--------------------------------------------------------------------------------
INDEX: 1
ID: 4006dbe5-63d2-41fc-b60c-f1ba255e0c8e
DOCUMENT: Basic Rules

- A match can be played as best of three or five sets.
- Each set consists of games, and each game consists of points.
- Points are scored as **0 (Love), 15, 30, 40**, and then **game**.
- A player must win a game by at least **two points**.
- The ball must land within the designated court boundaries.
EMBEDDING LENGTH: 384
FIRST 10 VALUES: [ 0.03660438  0.03123698 -0.01129173 -0.1105906  -0.03547212  0.08182574
 -0

In [58]:
# step 5 step up a llm pass a question this would be a text generation model 
pipe = pipeline("text-generation", model = "Qwen/Qwen2.5-1.5B-Instruct")


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

c:\Users\U S E R\github_projects\ai-engineering-lab\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\U S E R\.cache\huggingface\hub\models--Qwen--Qwen2.5-1.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [60]:
## step 6 retrieve and generation 
def retrieve_and_generate(query, threshold =1):
    """
    Retrieves relevant context from vector database and generates an answer get me top 1 answer 
    """
    search_results = vector_db.similarity_search_with_score(query, k=1)
    print(search_results)

    if not search_results or search_results[0][1] > threshold:
        return "I dont know the answer there is no avaliable context in vector db  "
    retrieved_context =  search_results[0][0].page_content
    similarity_score =  search_results[0][1]
    print(f"Similarity score: {similarity_score}")
    print(f"Retrieved context: {retrieved_context}")

    prompt = f"Answer the question using the given context \n Context: {retrieved_context} \n Question: {query} \n Answer: "
    response = pipe(prompt, max_new_tokens = 100)

    return response[0]["generated_text"]


In [61]:
question = " what are famous tournaments "
response = retrieve_and_generate(question)
print(response)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[(Document(metadata={}, page_content='Famous Tournaments\n\n- **Grand Slam Events**:\n- Australian Open\n- French Open\n- Wimbledon\n- US Open'), 0.6766194701194763)]
Similarity score: 0.6766194701194763
Retrieved context: Famous Tournaments

- **Grand Slam Events**:
- Australian Open
- French Open
- Wimbledon
- US Open


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Answer the question using the given context 
 Context: Famous Tournaments

- **Grand Slam Events**:
- Australian Open
- French Open
- Wimbledon
- US Open 
 Question:  what are famous tournaments  
 Answer:  Grand Slam Events, including the Australian Open, French Open, Wimbledon, and US Open. The Grand Slam events involve a single tournament held each year in which players compete for the highest prize money available. These events have been around since the late 19th century and are considered some of the most prestigious tennis competitions in the world. Players who win these events earn significant amounts of money and often become household names in the sport. Additionally, many professional tennis players consider winning a Grand Slam event to be


In [62]:
question =" what is cricket? "
response = retrieve_and_generate(question)
print(response)

[(Document(metadata={}, page_content="Introduction\n\nTennis is a popular sport played between two players (singles) or two teams of two players each (doubles). The game involves using a racket to hit a ball over a net into the opponent's court."), 1.1143043041229248)]
I dont know the answer there is no avaliable context in vector db  


In [63]:
question =" what is scoring system ? "
response = retrieve_and_generate(question)
print(response)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[(Document(metadata={}, page_content='Scoring System\n\n```plaintext\n0 points  -> Love\n1 point   -> 15\n2 points  -> 30\n3 points  -> 40\n4 points  -> Game (if leading by 2)\nDeuce     -> 40-40 (must win two consecutive points to win the game)\nAdvantage -> If a player wins a point at deuce, they gain the advantage\n```'), 0.7063989639282227)]
Similarity score: 0.7063989639282227
Retrieved context: Scoring System

```plaintext
0 points  -> Love
1 point   -> 15
2 points  -> 30
3 points  -> 40
4 points  -> Game (if leading by 2)
Deuce     -> 40-40 (must win two consecutive points to win the game)
Advantage -> If a player wins a point at deuce, they gain the advantage
```
Answer the question using the given context 
 Context: Scoring System

```plaintext
0 points  -> Love
1 point   -> 15
2 points  -> 30
3 points  -> 40
4 points  -> Game (if leading by 2)
Deuce     -> 40-40 (must win two consecutive points to win the game)
Advantage -> If a player wins a point at deuce, they gain the adv

In [64]:
question =" what is tennis? "
response = retrieve_and_generate(question)
print(response)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[(Document(metadata={}, page_content="Introduction\n\nTennis is a popular sport played between two players (singles) or two teams of two players each (doubles). The game involves using a racket to hit a ball over a net into the opponent's court."), 0.3389018177986145)]
Similarity score: 0.3389018177986145
Retrieved context: Introduction

Tennis is a popular sport played between two players (singles) or two teams of two players each (doubles). The game involves using a racket to hit a ball over a net into the opponent's court.
Answer the question using the given context 
 Context: Introduction

Tennis is a popular sport played between two players (singles) or two teams of two players each (doubles). The game involves using a racket to hit a ball over a net into the opponent's court. 
 Question:  what is tennis?  
 Answer:  Tennis is a sport that can be played by one player against another, or two teams against each other. It uses a special racket and a ball. The goal is to hit the ball 

In [84]:
question ="Which Equipment is used in tennis? "
response = retrieve_and_generate(question)
print(response)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[(Document(metadata={}, page_content='Equipment\n\n- **Racket**: Used to hit the ball.\n- **Tennis Ball**: Yellow-green in color, designed for optimal bounce.\n- **Court**: Can be grass, clay, or hard court.'), 0.6353908777236938)]
Similarity score: 0.6353908777236938
Retrieved context: Equipment

- **Racket**: Used to hit the ball.
- **Tennis Ball**: Yellow-green in color, designed for optimal bounce.
- **Court**: Can be grass, clay, or hard court.
Answer the question using the given context 
 Context: Equipment

- **Racket**: Used to hit the ball.
- **Tennis Ball**: Yellow-green in color, designed for optimal bounce.
- **Court**: Can be grass, clay, or hard court. 
 Question: Which Equipment is used in tennis?  
 Answer:  Tennis Ball and Racket are both equipment used in tennis.

Based on the information provided, which of these pieces of equipment is not typically used in tennis?
A) Racket
B) Tennis Ball
C) Court
D) None of the above

The answer is B) Tennis Ball. While a racket is 

In [85]:
## now let's perform manually dot product and see what happens in backed between query vector and stored vector db 
question ="Which Equipment is used in tennis? "
def embed_text(text):
    return embedding_model.encode(
        text,
        normalize_embeddings=True
    )
query_vector = embed_text(question)
print(query_vector)
print(len(query_vector))

[ 6.68694591e-03  3.57892402e-02 -2.27895156e-02 -1.02096006e-01
 -1.03357807e-01  1.10306242e-03  6.55323043e-02  4.13350388e-02
  8.46140236e-02  1.10619992e-01  1.75088237e-03  4.79849130e-02
 -2.88194492e-02  6.10498860e-02  3.61944698e-02 -2.40272600e-02
  5.86521849e-02  6.72812983e-02  2.20692009e-02 -1.37269469e-02
 -2.59693200e-03 -2.73796283e-02  1.95579473e-02 -3.00525129e-02
 -4.15579751e-02  5.97463083e-03 -5.05049713e-02  6.46979585e-02
  1.05819702e-02 -4.45339978e-02 -1.05188921e-01 -1.41142244e-02
 -1.54909818e-02  9.77852941e-03 -1.92788765e-01  4.51152958e-03
  2.20482647e-02  4.39524911e-02 -4.68352437e-02  6.56949505e-02
  2.57806350e-02 -7.09319338e-02  2.70782802e-02  6.07990175e-02
  4.96365950e-02  8.04069117e-02 -2.42100228e-02  5.45893572e-02
  2.42157523e-02 -1.70006342e-02 -6.49622008e-02  1.89477410e-02
  6.76276684e-02 -3.12167089e-02  9.96754095e-02 -2.77286749e-02
  2.11727954e-02  3.21195684e-02  5.11991642e-02  2.66399840e-03
  7.14313388e-02 -1.92164

In [78]:
## now let's perform manually dot product and see what happens in backed between query vector and stored vector db 
stored = vector_db._collection.get(
    include=["documents", "embeddings"]
)
doc_vector = stored["embeddings"][1]

print(stored["documents"][1])
print(len(doc_vector))
print(doc_vector[:10])

Basic Rules

- A match can be played as best of three or five sets.
- Each set consists of games, and each game consists of points.
- Points are scored as **0 (Love), 15, 30, 40**, and then **game**.
- A player must win a game by at least **two points**.
- The ball must land within the designated court boundaries.
384
[ 0.03660438  0.03123698 -0.01129173 -0.1105906  -0.03547212  0.08182574
 -0.01086429  0.01241309  0.0829439   0.05625681]


In [79]:
import numpy as np
query_np = np.array(query_vector)
doc_np = np.array(doc_vector)
dot_product = np.dot(query_np, doc_np)

print("Dot product:", dot_product)

Dot product: 0.38220253662526193


In [86]:
for i, doc_vector in enumerate(stored["embeddings"]):
    doc_vector = np.array(doc_vector)

    cosine = np.dot(query_np, doc_vector) / (
        np.linalg.norm(query_np) *
        np.linalg.norm(doc_vector)
    )

    print("\nDocument:", stored["documents"][i][:60])
    print("Cosine similarity:", cosine)


Document: Introduction

Tennis is a popular sport played between two p
Cosine similarity: 0.6397533534120197

Document: Basic Rules

- A match can be played as best of three or fiv
Cosine similarity: 0.3822025771443706

Document: Scoring System

```plaintext
0 points  -> Love
1 point   -> 
Cosine similarity: 0.2272069203120514

Document: Famous Tournaments

- **Grand Slam Events**:
- Australian Op
Cosine similarity: 0.33638657744944916

Document: Equipment

- **Racket**: Used to hit the ball.
- **Tennis Ba
Cosine similarity: 0.6380786242374951

Document: Conclusion

Tennis is a thrilling sport that requires skill,
Cosine similarity: 0.581068034780388


In [ ]:
## so in dot product higher the value more is the similarity and in vector db similarity with score method lesser the value more similarity as it gives distance 